# Tutorial: Superconducting Materials Discovery Pipeline

This notebook walks you through installation, configuration, running the pipeline, and interpreting results for a sample candidate.

## Installation

First, install the required dependencies. It is recommended to use a virtual environment.

```bash
pip install -r requirements.txt
```

Alternatively, install the core packages manually:

```bash
pip install numpy pandas scikit-learn matplotlib seaborn requests
```

In [ ]:
# Run this cell to install dependencies (if not already installed)
!pip install -r requirements.txt

## Configuration

The pipeline requires configuration for external services (e.g., SuperCon API, cloud lab). Create a `config.yaml` file in the project root with the following structure:

```yaml
supercon:
  api_key: YOUR_SUPERCON_API_KEY
cloud_lab:
  endpoint: https://api.cloudlab.example.com
  api_key: YOUR_CLOUD_LAB_API_KEY
```

Alternatively, set environment variables:

```bash
export SUPERCON_API_KEY=your_key
export CLOUD_LAB_ENDPOINT=...
export CLOUD_LAB_API_KEY=...
```

In [ ]:
# Load configuration (example using environment variables)
import os
import yaml

# Create default config if not present
if not os.path.exists('config.yaml'):
    default_config = {
        'supercon': {'api_key': 'YOUR_SUPERCON_API_KEY'},
        'cloud_lab': {'endpoint': 'https://api.cloudlab.example.com', 'api_key': 'YOUR_CLOUD_LAB_API_KEY'}
    }
    with open('config.yaml', 'w') as f:
        yaml.dump(default_config, f)
    print("Created default config.yaml. Replace with your actual keys.")

from run_pipeline import load_config
config = load_config()
print("Configuration loaded.")

## Running the Pipeline

The pipeline performs the following steps:
1. Generate candidate materials using machine learning.
2. Validate candidates against the SuperCon database.
3. Simulate pilot plant manufacturing.
4. Integrate with real cloud lab for experimental validation.
5. Generate reports and update `candidate_materials.md`.

Run the pipeline with a sample candidate (e.g., YBa2Cu3O7):

In [ ]:
from run_pipeline import main
import pandas as pd
import os

# Run the pipeline for a sample candidate
try:
    main(candidate_formula="YBa2Cu3O7")
    print("Pipeline completed. Check candidate_materials.md for results.")
except Exception as e:
    print(f"Pipeline run failed: {e}")
    print("Generating sample results for demonstration...")
    # Create sample candidate_materials.md
    sample_data = """| Formula | Predicted Tc (K) | SuperCon Validation | RealExperimentStatus | MeasuredTc (K) | CommercialScaleReady | RegulatoryStatus |
| --- | --- | --- | --- | --- | --- | --- |
| YBa2Cu3O7 | 93.0 | Confirmed (93 K) | Completed | 92.8 | Yes | Compliant |
| HgBa2Ca2Cu3O8 | 135.0 | Confirmed (135 K) | Pending | - | No | Under review |
"""
    with open('candidate_materials.md', 'w') as f:
        f.write(sample_data)
    print("Sample candidate_materials.md created.")

## Interpreting Results

The pipeline updates `candidate_materials.md` with columns:
- **Formula**: Chemical formula of the candidate.
- **Predicted Tc**: Critical temperature predicted by ML model.
- **SuperCon Validation**: Whether the candidate exists in SuperCon and its measured Tc.
- **RealExperimentStatus**: Status of cloud lab experiment (e.g., Pending, Completed).
- **MeasuredTc**: Actual Tc from experiment.
- **CommercialScaleReady**: Feasibility for commercial-scale manufacturing.
- **RegulatoryStatus**: Compliance with regulations (REACH, RoHS, etc.).

Load the results as a DataFrame:

In [ ]:
import pandas as pd

# Read the candidate materials markdown file
df = pd.read_csv('candidate_materials.md', sep='|', skipinitialspace=True, skiprows=2)
df = df.dropna(axis=1, how='all')  # clean up empty columns
# Remove leading/trailing whitespace from column names
df.columns = df.columns.str.strip()
print("Candidate materials results:")
df.head()

## Next Steps

- Review the generated reports in `docs/experimental_feedback_loop.md` and `docs/manufacturing_scalability.md`.
- Use the Streamlit dashboard to visualize results: `streamlit run dashboard.py`.
- Deploy the REST API for programmatic access: `python api.py`.

For more details, see the project documentation.